In [1]:
import sqlite3
import pandas as pd

connection = sqlite3.connect("../data/sk_hynix.db")

with open ("../sql/event_window.sql") as f:
    query1 = f.read()

print(query1)
event_window = pd.read_sql_query(query1,connection)
event_window

SELECT 
    e.event_id,
    e.event_date,
    e.event_description,
    e.event_type,
    p.close AS price_on_event_day,
    LAG(p.close, 3) OVER (ORDER BY p.date) AS price_3days_before,
    LEAD(p.close, 3) OVER (ORDER BY p.date) AS price_3days_after
FROM events e
JOIN daily_price p ON e.event_date = p.date
ORDER BY e.event_date;


,event_id,event_date,event_description,event_type,price_on_event_day,price_3days_before,price_3days_after
0,E01,2026-01-30,"""SK Hynix rose 6.2% to 949,000 won ($656.29), ...",market_volatility,909000,NaN,1225000.0
1,E02,2026-02-20,"""SK hynix shares gain traction after BlackRock...",foreign_flow,949000,NaN,1970000.0
2,E03,2026-02-24,"Intraday break above 1,000,000 won ($)",foreign_flow,1005000,NaN,2919000.0
3,E04,2026-04-23,"SK Hynix posts record first-quarter profit, in...",earnings,1225000,909000.0,2580000.0
4,E05,2026-05-14,"First-ever break above 2,000,000 won ($) in pr...",market_volatility,1970000,949000.0,2180000.0
5,E06,2026-06-22,"""SK Hynix overtakes Samsung to become South Ko...",market_volatility,2919000,1005000.0,1845000.0
6,E07,2026-06-24,"""South Korea’s biggest chipmaker SK Hynix plan...",corporate_action,2580000,1225000.0,NaN
7,E08,2026-07-10,"""South Korea’s SK Hynix raises $26.5bn in reco...",corporate_action,2180000,1970000.0,NaN
8,E09,2026-07-13,"SK Hynix South Korean shares clock worst day, ...",market_volatility,1845000,2919000.0,NaN


In [8]:
# 쿼리 2 실행
with open("../sql/excess_return.sql") as f:
    query2 = f.read()
excess = pd.read_sql_query(query2, connection)
excess.head()

,date,stock_return,kospi_return,excess_return
0,2026-01-02,NaN,NaN,NaN
1,2026-01-05,2.806499,3.431617,-0.625118
2,2026-01-06,4.310345,1.524615,2.785730
3,2026-01-07,2.203857,0.565244,1.638613
4,2026-01-08,1.886792,0.028785,1.858008


In [11]:

def get_excess_return_window(event_date, window=3):
    idx = excess[excess['date'] == event_date].index
    if len(idx) == 0:
        return None
    i = idx[0]
    window_data = excess.iloc[max(0, i-window):i+window+1]
    return window_data['excess_return'].sum()

event_window['cumulative_excess_return'] = event_window['event_date'].apply(get_excess_return_window)
event_window[['event_id', 'event_description', 'event_type', 'cumulative_excess_return']]

,event_id,event_description,event_type,cumulative_excess_return
0,E01,"""SK Hynix rose 6.2% to 949,000 won ($656.29), ...",market_volatility,12.980748
1,E02,"""SK hynix shares gain traction after BlackRock...",foreign_flow,4.346270
2,E03,"Intraday break above 1,000,000 won ($)",foreign_flow,6.676448
3,E04,"SK Hynix posts record first-quarter profit, in...",earnings,7.459371
4,E05,"First-ever break above 2,000,000 won ($) in pr...",market_volatility,7.468623
5,E06,"""SK Hynix overtakes Samsung to become South Ko...",market_volatility,19.385287
6,E07,"""South Korea’s biggest chipmaker SK Hynix plan...",corporate_action,6.834793
7,E08,"""South Korea’s SK Hynix raises $26.5bn in reco...",corporate_action,-0.474086
8,E09,"SK Hynix South Korean shares clock worst day, ...",market_volatility,-4.479968


In [14]:
summary = event_window.groupby('event_type')['cumulative_excess_return'].agg(['mean', 'std', 'count'])
summary.sort_values('mean', ascending=False)

,mean,std,count
event_type,,,
market_volatility,8.838673,10.126716,4
earnings,7.459371,NaN,1
foreign_flow,5.511359,1.647684,2
corporate_action,3.180354,5.168158,2


In [15]:
event_window.to_csv("../data/processed/event_window_results.csv", index=False)
summary.to_csv("../data/processed/event_type_summary.csv")